# Python и интернет, часть 2: как скачать всё отовсюду

In [ ]:
%pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin # для удобной работы со ссылками, импортируем отдельный метод
from pprint import pprint # для красивого вывода

На прошлой паре мы изучали общее устройство интернета, связь клиент-сервер, возможные проблемы при передаче данных. Мы уже обсудили, как получать ответ сервера через python с помощью модуля `requests` и как устроен язык разметки html.

In [ ]:
# краткий пересказ

response = requests.get('https://habr.com/ru/') # скачиваем нужную страницу
print('Статус:', response.status_code, sep='\n') # код статуса, получили ли мы ожидаемую страницу
print('\nКод страницы:', response.text[:100], sep='\n') # текст (html-код) страницы, первые символы

Статус:
200

Код страницы:
<!DOCTYPE html>
<html lang="ru">

  <head>
    <title>Публикации &#x2F; Моя лента &#x2F; Хабр</title


На этот раз мы будем изучать, как именно достать необходимые данные из html-кода и как правильно автоматизировать сбор данных с сайта (и при этом не уронить сайт и не попасться).

## BeautifulSoup: достаем данные из HTML

BeautifulSoup разбирает HTML в дерево элементов. Такой разбор называют **парсингом**. После него можно искать теги, читать атрибуты и извлекать текст, не вырезая куски из исходной строки вручную.

Создадим `soup` из учебной строки `html_page`. Второй аргумент, `'html.parser'`, указывает встроенный в питон механизм разбора HTML (для скорости на больших данных вместо html.parser можно использовать [lxml](https://beautiful-soup-4.readthedocs.io/en/latest/#installing-a-parser)). Для скачанной страницы вместо `html_page` передают `response.text`.

[Документация BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

In [ ]:
html_page = """<!DOCTYPE html>
<html lang="ru">
  <head>
    <meta charset="utf-8">
    <title>Новости семинара</title>
  </head>
  <body>
    <h1 id="top">Питон и интернет</h1>
    <p class="intro">Сегодня учимся доставать <b>данные</b> из HTML.</p>
    <!-- Комментарий: этот текст не показывается на странице -->
    <article class="post" id="post-1">
      <h2><a href="/news/1">Открылась запись на семинар</a></h2>
      <p class="author">Анна</p>
      <time datetime="2026-11-01">1 ноября</time>
    </article>
    <article class="post" id="post-2">
      <h2><a href="/news/2">Новая подборка материалов</a></h2>
      <p class="author">Мария</p>
      <time datetime="2026-11-02">2 ноября</time>
    </article>
    <a href="#top">Наверх</a>
    <br>
  </body>
</html>"""

In [ ]:
soup = BeautifulSoup(html_page, 'html.parser') # в soup будет лежать "обработанная" страница, по которой можно искать и т.д.
print(soup.get_text())





Новости семинара


Питон и интернет
Сегодня учимся доставать данные из HTML.


Открылась запись на семинар
Анна
1 ноября


Новая подборка материалов
Мария
2 ноября

Наверх





Итак, мы получили весь текст со страницы. Но:
1) Мы не вполне понимаем, какой из фрагментов к чему относится. Что из этого было заголовком, что названиями статей? Можно придумывать методы определения, но легче было сразу забрать то, что нам было нужно, по соответствующим тегам.
2) Нам может потребоваться метаинформация: ссылки, даты и прочие параметры, хранящиеся внутри тегов, но не в прямом текстовом виде на странице.

Так что продолжаем разбираться с BeautifulSoup поподробнее.

### Найти один элемент

Метод **`find`** возвращает первый подходящий элемент. Например, `soup.find('article', {'class': 'post'})` ищет тег `article` с классом `post`.

С найденным элементом можно работать дальше:

- `prettify()` - показать его разметку с отступами;
- `get_text()` - извлечь текст;
- `find(...)` - найти другой элемент внутри него.

In [ ]:
post = soup.find('article', {'class': 'post'})
print(post.prettify())
print('Только текст:', post.get_text(' ', strip=True))

<article class="post" id="post-1">
 <h2>
  <a href="/news/1">
   Открылась запись на семинар
  </a>
 </h2>
 <p class="author">
  Анна
 </p>
 <time datetime="2026-11-01">
  1 ноября
 </time>
</article>

Только текст: Открылась запись на семинар Анна 1 ноября


В `get_text(' ', strip=True)` пробел задает разделитель между текстовыми фрагментами, а `strip=True` убирает пробелы по их краям. Без разделителя слова из соседних тегов могут склеиться.

Если элемент не найден, `find` возвращает **`None`**. Вызвать у него `get_text()` нельзя, поэтому для необязательных элементов сначала проверяем результат поиска:

In [ ]:
subtitle = soup.find('h3')
if subtitle is None:
    print('Подзаголовка h3 на странице нет')
else:
    print(subtitle.get_text(strip=True))

Подзаголовка h3 на странице нет


### Найти все элементы

Первую новость мы нашли через `find`. Для всех новостей нужен **`find_all`**: он возвращает коллекцию элементов, по которой можно пройти циклом. Если совпадений нет, коллекция будет пустой.

Внутри каждого найденного `article` достанем заголовок и автора:

In [ ]:
posts = soup.find_all('article', {'class': 'post'})
print('Новостей:', len(posts), '\n')

for post in posts:
    title = post.find('h2')
    author = post.find('p', {'class': 'author'})
    print(title.get_text(strip=True))
    print(author.get_text(strip=True))
    print('-- ' * 10)  # для красоты

Новостей: 2 

Открылась запись на семинар
Анна
-- -- -- -- -- -- -- -- -- -- 
Новая подборка материалов
Мария
-- -- -- -- -- -- -- -- -- -- 


Обратите внимание, что мы только что совершили первый шаг к автоматическому сбору датасета: заголовок новости и автор хранятся в нашем коде в отдельных переменных, и мы могли бы, например, не печатать эту информацию с помощью `print`, а добавлять строки в датафрейм со столбцами "пост" и "автор".

### Достать атрибуты и построить полный адрес

Достанем ссылку первой новости. Ее текст получаем через `get_text()`, а атрибуты смотрим в словаре `.attrs`.

Адрес находится в атрибуте `href`. Запись `['href']` подойдет, если он точно есть. Если атрибут может отсутствовать, используйте `.get('href')`: вместо ошибки получится `None`.

In [ ]:
title_obj = posts[0].find('a')
print('Текст ссылки:', title_obj.get_text(strip=True))
print('Атрибуты:', title_obj.attrs)
href = title_obj.attrs['href']
print('Адрес из HTML:', href)

Текст ссылки: Открылась запись на семинар
Атрибуты: {'href': '/news/1'}
Адрес из HTML: /news/1


Получили `/news/1`. Чтобы превратить эту относительную ссылку в полный адрес, используем **`urljoin`** из `urllib.parse`. Первым аргументом передадим адрес исходной страницы, вторым - найденную ссылку.

Функция учитывает ссылки с `/`, с `../` и уже полные адреса. Для скачанной страницы за основу обычно берут `response.url`, где учтены перенаправления.

In [ ]:
from urllib.parse import urljoin

base_url = 'https://example.com/'
print(urljoin(base_url, href))
print(urljoin(base_url, 'https://other.example/news/2')) # вот что будет, если ссылка окажется не относительной
print('Дата первой новости:', posts[0].find('time').get('datetime'))

https://example.com/news/1
https://other.example/news/2
Дата первой новости: 2026-11-01


### Другой способ поиска: CSS-селекторы

**CSS-селектор** - короткая запись условия поиска элементов. В CSS такие условия задают, к чему применить оформление. BeautifulSoup использует их для поиска.

| Селектор | Что выбирает |
| --- | --- |
| `article` | Все теги article |
| `.post` | Элементы с классом post |
| `#post-2` | Элемент с id post-2 |
| `article.post` | Теги article с классом post |
| `article.post h2 a` | Все ссылки (тег 'а'), находящиеся внутри h2 внутри article с классом post |

Точка обозначает класс, решетка - id, пробел - вложенность на любой глубине.

**`select`** возвращает все совпадения, **`select_one`** - первое или `None`. Например, `select_one('#post-2')` найдет ту же новость, что и `find(id='post-2')`.

In [ ]:
for link in soup.select('article.post h2 a'):
    print(link.get_text(strip=True), link.get('href'))

print('Автор второй новости:', soup.select_one('#post-2 .author').get_text(strip=True))

Открылась запись на семинар /news/1
Новая подборка материалов /news/2
Автор второй новости: Мария


### Возвращаемся к настоящей странице: заголовки Хабра

Теперь вместо учебной строки возьмем ответ Хабра из `response.text`. Разберем его в отдельный `habr_soup`, чтобы сохранить учебный пример в `soup`.

В текущей разметке заголовок публикации - это тег `h2` с классами `tm-title` и `tm-title_h2`. Найдем первый пост, выведем его HTML и текст:

In [ ]:
response = requests.get('https://habr.com/ru/', timeout=10)
habr_soup = BeautifulSoup(response.text, 'html.parser')
post = habr_soup.find('h2', {'class': 'tm-title tm-title_h2'})

if post is None:
    print('Ожидаемый заголовок не найден. Нужно проверить ответ и разметку сайта.')
else:
    print(post.prettify())
    print(post.get_text(' ', strip=True))

<h2 class="tm-title tm-title_h2" data-v-31a09e03="">
 <!--[-->
 <a class="tm-title__link" data-article-link="true" href="/ru/articles/1105154/">
  <span>
   Качественная анимация на микроконтроллере с оптимизацией, через мои костыли:
  </span>
 </a>
 <!--]-->
</h2>

Качественная анимация на микроконтроллере с оптимизацией, через мои костыли:


Первый заголовок получили. Найдем все через `find_all` и выведем первые три вместе со ссылками на статьи. Здесь пригодится `urljoin`: в HTML Хабра ссылки относительные.

In [ ]:
habr_titles = habr_soup.find_all('h2', {'class': 'tm-title tm-title_h2'})
print('Найдено заголовков:', len(habr_titles))

for post in habr_titles[:3]:
    link = post.find('a')
    print(post.get_text(' ', strip=True))
    if link is not None and link.get('href'):
        print(urljoin(response.url, link.get('href')))
    print('-- ' * 10)  # для красоты

Найдено заголовков: 18
Качественная анимация на микроконтроллере с оптимизацией, через мои костыли:
https://habr.com/ru/articles/1105154/
-- -- -- -- -- -- -- -- -- -- 
Агент написал верный код, а потом удалил его ради скрытых тестов: что показал MaxxBench
https://habr.com/ru/articles/1105152/
-- -- -- -- -- -- -- -- -- -- 
Ее руководитель мог стать архитектором в Нью-Йорке. Что такое S-Game и почему Phantom Blade Zero порвала рынок
https://habr.com/ru/companies/timeweb/articles/1102776/
-- -- -- -- -- -- -- -- -- -- 


---

## Шпаргалка: от адреса до данных

```python
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1. Получить ответ и проверить HTTP-статус.
response = requests.get('https://habr.com/ru/', timeout=10)
response.raise_for_status()

# 2. Разобрать именно тот HTML, который пришел в ответе.
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Найти нужные элементы и извлечь значения.
for heading in soup.select('article h2')[:3]:
    link = heading.find('a')
    if link is not None and link.get('href'):
        print(link.get_text(' ', strip=True))
        print(urljoin(response.url, link.get('href')))
```

Если результат неожиданный, проверяйте шаги по порядку: ответ и статус, полученный HTML, затем условия поиска элементов.

## Небольшая практика

### 1. Заголовок, автор и время публикации

Для первых трех публикаций Хабра соберите список словарей с ключами:

- `title` - заголовок;
- `url` - полный адрес статьи;
- `author` - никнейм автора;
- `published_at` - время из атрибута `datetime`.
<br>

**ПОДСКАЗКИ**

Используйте уже полученный `habr_soup`:

1. Найдите первые три `article`.
2. Внутри каждого найдите заголовок и ссылку, автора и время.
3. Извлеките значения и добавьте словарь в список.

Автор в текущей разметке находится в ссылке с классом `tm-user-info__username`, время - в теге `time`. Если автора или времени нет, сохраните `None`.

Для времени нужен полный `datetime`, а не текст «11 минут назад». Сверьте одну полученную запись с HTML статьи.

Переведите полученные данные в датафрейм pandas (лучше работать со списком словарей, чем записывать данные в датафрейм построчно, на больших данных построчное добавление будет работать ужасно медленно) - [нужный метод](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.from_records.html)

In [ ]:
# Посмотрим на начало разметки первой публикации.
article = habr_soup.find('article')
if article is not None:
    print(article.prettify()[:3500])

<article class="tm-articles-list__item" data-navigatable="" id="1102946" tabindex="0">
 <div class="article-snippet" data-v-31a09e03="">
  <!--[-->
  <div class="publication-type-label publication-type-label_type-article">
   <span class="publication-type-label__label publication-type-label__label_type-article">
    Статья
   </span>
  </div>
  <!--]-->
  <div class="meta-container" data-v-31a09e03="">
   <div class="meta" data-v-31a09e03="">
    <span class="tm-user-info author" data-v-31a09e03="">
     <a class="tm-user-info__userpic" href="/ru/users/wbb/">
      <!--[-->
      <div class="tm-entity-image">
       <!--[-->
       <img alt="" class="tm-entity-image__pic" height="24" loading="lazy" src="https://assets.habr.com/habr-web/release_2.357.1/client/img/avatars/116.png" width="24"/>
       <!--]-->
      </div>
      <!--]-->
     </a>
     <span class="tm-user-info__user tm-user-info__user_appearance-default">
      <a class="tm-user-info__username" href="/ru/users/wbb/">
   

In [ ]:
# Ваш код: список словарей для первых трех публикаций.

**Доп. практика:**

Скачайте [учебную страницу погоды](https://simple-weather-website.netlify.app/) и получите город, температуру, влажность и скорость ветра.

1. Отправьте один запрос и проверьте статус.
2. Разберите HTML через BeautifulSoup.
3. Найдите элементы с классами `city`, `temp`, `humidity` и `wind`.
4. Выведите значения одной строкой, сохранив единицы измерения.

Это демонстрационный сайт. В исходном HTML записаны учебные значения, которые JavaScript может заменить после загрузки. В задании извлекаем данные ответа сервера, а не актуальный прогноз.

\+ уберите единицу измерения из строки температуры и превратите число в `float`. Если значения в браузере и питоне различаются, объясните, откуда взялось различие.

## Краулеры

In [ ]:
!pip install fake_useragent

In [ ]:
import time
from datetime import datetime
from fake_useragent import UserAgent
import random

Теперь предположим, что мы хотим скачать не только заголовки, но и сами посты. Для этого мы хотим:
- пройтись циклом по нескольким страницам поиска (например, мы хотим обработать последние 100 постов, а на странице только 20)
- сохранить все ссылки на посты
- получить каждую из страниц и достать с неё нужные данные.

Однако **очень важно** узнать о *некоторых подводных камнях* до того, как сайт заблокирует вас на 5 часов :)

### Что вообще такое краулеры?

Краулеры $-$ это боты / программы, которые "ползают" (*crawl*) по страницам сайта и собирают информацию. Все чаще использование таких программ запрещается правилами пользования сайтами, поэтому формально это не всегда хорошо. Но так продолжают делать, и это полезно уметь. Запрещают по 2 основным причинам: не хотят делиться данными или боятся, что вы уроните сайт (если сервер слабый, то это довольно легко).

Поэтому нужно собирать данные аккуратно, чтобы:
1. вас не заблокировали по IP,
2. вы не навредили серверу.

Вместо отдельных запросов для этого лучше создать сессию, которая позволит хранить информацию между запросами и поддерживать то же соединение и не создавать каждый раз все заново, что влияет на производительность.

In [ ]:
session = requests.session()

Теперь для запроса вместо requests.get мы пишем session.get.

### Правило 0: здоровый сон

Если вы обкачиваете много страниц подряд, ставьте `time.sleep(x)`, чтобы между запросами была задержка в x секунд. Это прежде всего важно, чтобы не уронить сервер, особенно если вы работаете с небольшим сайтом: представьте себе удивление региональной газеты, когда 80 студентов ФиКЛа пытаются быстренько получить все её новости за час до дедлайна...

In [ ]:
time.sleep(3)

Полезно это и для маскировки под реальное пользовательское поведение: регулярные запросы раз в полсекунды вполне могут показаться серверу подозрительными, даже если в остальном вы успешно замаскировались:

In [ ]:
time.sleep(random.uniform(3.1, 5.2))

Обратите внимание, за счёт этой паузы между запросами обкачивание сайта может занять достаточное время. Не стоит уменьшать это время, чтобы код побыстрее загрузился, если не уверены, что сайт это выдержит и при этом не сочтёт вас подозрительным.

### Правило 1: представиться браузером

In [ ]:
response = session.get('https://www.avito.ru/')
response

<Response [403]>

403 $-$ сервер понял запрос, но отказался выдавать нам страницу. Почему?

Когда Python-библиотека подключается к сайту, она проходит процедуру TLS-рукопожатия (см. предыдущую тетрадку). У Python и у настоящего браузера этот процесс выглядит по-разному: разные шифры, разный порядок параметров. Наконец, по умолчанию Python честно представляется собой, а не браузером, а многие сайты пытаются защитить свои данные от обкачивания. Попробуем замаскироваться: при подключении Python будет представляться серверу браузером.

In [ ]:
ua = UserAgent()
headers = {'User-Agent': ua.random}
# альтернативный способ
# headers = {'User-Agent': ua.chrome}
print(headers)

{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'}


In [ ]:
response = session.get('https://www.avito.ru/', headers=headers)
response

<Response [439]>

В случае с Авито это не поможет: далеко не всем сайтам достаточно "имени", под которым мы заходим, они смотрят и на другие параметры.

### Остальные правила

1. **Параметры запроса**: помимо UserAgent полезно настроить и другие `headers`, а также передать cookies. Можно зайти на сайт вручную, без питона, открыть devtools и получить [здесь](https://curlconverter.com/) готовый запрос на python (главное, не храните ваши персональные данные в тетрадке! многие из этих не представляют большой ценности, но хорошо завести привычку хранить все подобные идентификаторы в отдельных файлах .env)
2. **Прокси**: если вы работаете через Colab, то при блокировке IP можно попробовать перезагрузить сессию (`disconnect and delete runtime`, не забудьте сохранить все файлы, которые вам дороги). Однако некоторые сайты в целом блокируют семейство облачных адресов и заметят любую попытку получить страницу с помощью Colab, или же вас могут заблокировать при работе через локальное приложение (Jupyter Notebook, VS Code). В таком случае полезно использовать [прокси](https://requests.readthedocs.io/en/latest/user/advanced/#proxies). Можно смотреть на [некоторых сайтах](https://checkerproxy.net/) бесплатные, хотя они обычно очень недолговечны.
4. **JavaScript**: requests не работает с динамическими сайтами, которые загружаются через JS, но для этого есть более хитрые библиотеки: Selenium, PlayWright и другие. Обзор существующих инструментов есть, например, [здесь](https://oxylabs.io/blog/python-web-scraping-libraries#1).

### Стратегии сбора данных



По сути краулеры выполняют последовательный сбор страниц с помощью циклов. Можно выделить разные стратегии сбора данных:
    
**По типу навигации**

1. Все страницы со ссылками имеют удобные номера ("https://ficbook.net/fanfiction/no_fandom/originals?p=2"), обычно просто `p=<число>` или `page=<число>`. В этом случае вам нужно просто подставлять цифры подробнее про параметры передаваемые в ссылке можно посмотреть [здесь](https://en.wikipedia.org/wiki/Query_string).
2. Страницы называются как-то не структурированно (например, по названиям блоков). Тут нужно собирать ссылки на эти страницы (например, найти сбоку меню или список разделов и ссылки на них) и потом по ним ходить и собирать конечные странички.
3. Все расположено на одной страничке и догружается с использованием [WebSocket](https://en.wikipedia.org/wiki/WebSocket) или других технологий, при этом адрес в адресной строке никак не изменяется, данные могут догружаться на сайт автоматически по мере скролла страницы. Такие страницы не скачиваются стандартными методами, которым мы учимся сейчас, но это возможно (см. список доп. методов)

**По скорости обновления**

1. Если сайт довольно статичный по контенту (медленно появляются и удаляются материалы), то можно сначал собрать ссылки, а потом по ним ходить.
2. Если сайт очень динамичный по контенту (например, объявления на крупном сайте), вам нужно при получении ссылок с одной странички сразу их обходить, а потом переходить к следующей, потому что ко времени получения исчерпывающего списка ссылок по сайту многие будут уже удалены или недоступны. Также нужно настроить проверку, чтобы не скачивать одну и ту же информация несколько раз (если пока вы обрабатывали первую страницу со ссылками, добавились ещё данные, и теперь часть знакомых ссылок окажется на второй странице, когда вы до неё дойдёте)

### **Общие советы**

Базовый алгоритм:
- понять, как устроены страницы с выдачей результатов (новостей, постов, отзывов)
- пройти циклом по страницам и сохранить список ссылок
- пройти циклом по ссылкам и сохранить нужную информацию

1. Сохраняйте промежуточные результаты: список ссылок на отдельные страницы (постов, новостей, etc.), скачанный html-код страниц. Если у вас получится сохранить сырые html-данные, дальше вы можете спокойно работать с ними не боясь блокировок или ситуации, когда вы сохранили из страницы не все необходимые параметры и приходится запускать краулер заново.
2. Используйте библиотеку [tqdm](https://tqdm.github.io/), чтобы следить за прогрессом.
3. Если сайт блокирует вас в процессе, то лучше а) заранее иметь список ссылок, б) настроить поведение при блокировке (повторить запрос / остановить выполнение / etc.), в) сохранять обкачанное, даже если успела скачаться только часть, потом эти данные можно объединить (только внимательно следить, что в данных не получилось дубликатов)
4. Настраивайте базовые параметры против блокировок сразу, не проверяйте, засечёт вас сайт без них или нет, чтобы не пришлось ждать несколько часов, пока сайт про вас не забудет
<br>
<br>

**Удачи.**